# @localtester/wind-curve

Wind farm power from a turbine power curve.

Run all cells: **Runtime → Run all** (or ⌘/Ctrl + F9). The last code cell prints the endpoint address.


In [ ]:
PRIMITIVE = "@localtester/wind-curve"
ARCHIVE = "https://0303-14-201-56-230.ngrok-free.app/api/artifacts/localtester/wind-curve/download-archive/"
WORKDIR = "/content/primitive"

import shutil, urllib.request, zipfile
from pathlib import Path

shutil.rmtree(WORKDIR, ignore_errors=True)          # re-runs start clean
Path(WORKDIR).mkdir(parents=True, exist_ok=True)

archive = Path("/content/primitive.zip")
urllib.request.urlretrieve(ARCHIVE, archive)
with zipfile.ZipFile(archive) as zf:
    zf.extractall(WORKDIR)

files = sorted(p for p in Path(WORKDIR).rglob("*") if p.is_file())
print(f"{PRIMITIVE}: {len(files)} files in {WORKDIR}")
for p in files[:20]:
    print(" ", p.relative_to(WORKDIR))
if len(files) > 20:
    print(f"  … and {len(files) - 20} more")


In [ ]:
import subprocess, sys
from pathlib import Path

req = next(Path(WORKDIR).rglob("requirements.txt"), None)
if req is None:
    print("No requirements.txt. Install what you need with pip, e.g. !pip install pandas")
else:
    print(f"Installing {req.relative_to(WORKDIR)} …")
    done = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
        capture_output=True, text=True,
    )
    print("done" if done.returncode == 0 else f"pip failed:\n{done.stderr[-2000:]}")


In [ ]:
import os, re, subprocess, sys, time, urllib.request

PORT = 8011
MODULE = "powerai_endpoint"
CLOUDFLARED = "/usr/local/bin/cloudflared"

# Re-running this cell has to be safe: a previous run leaves a server and a tunnel
# alive, and the next one would otherwise pass the health probe anyway because the
# *stale* server answers it — a stale process serving stale code, reported as fresh.
for pattern in ("cloudflared tunnel", f"uvicorn {MODULE}:app"):
    subprocess.run(["pkill", "-f", pattern], capture_output=True)
time.sleep(1)

if not os.path.exists(CLOUDFLARED):
    got = subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
        f"cloudflared-linux-amd64 -O {CLOUDFLARED} && chmod +x {CLOUDFLARED}",
        shell=True, capture_output=True, text=True,
    )
    if got.returncode != 0:
        raise RuntimeError(f"could not install cloudflared:\n{got.stderr[-1000:]}")
    print("cloudflared installed")
else:
    print("cloudflared already present")

# --app-dir puts WORKDIR on the import path for this one process, so the archive
# need not be pip-installed (and may not even be a package) to be served.
# A src/ layout keeps its packages one folder down; both folders go on the path.
env = dict(os.environ, PYTHONPATH=os.pathsep.join(
    [WORKDIR, os.path.join(WORKDIR, "src"), os.environ.get("PYTHONPATH", "")]
))
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", f"{MODULE}:app", "--app-dir", WORKDIR,
     "--host", "127.0.0.1", "--port", str(PORT)],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env,
)

for _ in range(90):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=1)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(
        f"{MODULE}.py did not come up as a healthy server on :{PORT} — check its "
        "own errors above; this is the archive's own code, not the Hub's"
    )
print(f"endpoint up on :{PORT}")

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
PUBLIC_URL = ""
deadline = time.time() + 90
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        break
    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if found:
        PUBLIC_URL = found.group(0)
        break
if not PUBLIC_URL:
    raise RuntimeError("no tunnel URL; re-run this cell")

ENDPOINT_URL = PUBLIC_URL + "/run"
print()
print("Endpoint address:")
print("   ", ENDPOINT_URL)


Copy the endpoint address above and paste it into the agent chat on [the Agent tab](http://localhost:5173/localtester/wind-curve/agent).